In [0]:
# sanity: should print 1 row
spark.range(1).show()


+---+
| id|
+---+
|  0|
+---+



In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS de_genai")
spark.sql("USE de_genai")
print("Using DB:", spark.sql("SELECT current_database()").collect()[0][0])


Using DB: de_genai


In [0]:
from pyspark.sql import functions as F
from datetime import datetime, timedelta
import uuid, random

N = 1000  # you can increase later

tenants = ["alpha", "beta", "gamma"]
channels = ["email","chat","phone","portal"]
langs = ["en","ur","ar"]

subjects = [
  "Login issue","Payment failed","App crash on launch",
  "Feature request","Account locked","Refund request","Slow performance"
]
bodies = {
  "en": ["User reports timeout when submitting forms.",
         "Payment fails after retry.",
         "App crashes after update on Android 14.",
         "Customer asks for SSO integration."],
  "ur": ["App open nahi ho rahi.","Payment ke baad confirmation nahi aaya.",
         "Login par error aata hai."],
  "ar": ["بطء شديد في التطبيق.","فشل الدفع ولا توجد رسالة تأكيد.",
         "أخطاء عند تسجيل الدخول."]
}

rows=[]
now = datetime.utcnow()
for _ in range(N):
    ts = now - timedelta(minutes=random.randint(0, 60*24*7))
    lang = random.choice(langs)
    rows.append((
        random.choice(tenants),
        str(uuid.uuid4()),
        random.choice(subjects),
        random.choice(bodies[lang]),
        random.choice(channels),
        ts.isoformat()+"Z",
        lang
    ))

schema = "tenant_id string, ticket_id string, subject string, body string, channel string, created_ts string, lang string"
df = spark.createDataFrame(rows, schema)
df = df.withColumn("event_date", F.to_date("created_ts"))
df.limit(5).display()


/home/spark-00f8ecd1-440c-45db-be28-c2/.ipykernel/2445/command-6768248463866592-2101825374:27: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow()


tenant_id,ticket_id,subject,body,channel,created_ts,lang,event_date
alpha,909e6202-0737-4b49-8c80-bc773413fec7,Refund request,بطء شديد في التطبيق.,chat,2025-08-28T08:33:01.104112Z,ar,2025-08-28
alpha,402f92ae-72cf-4312-9451-ce7bcc713082,Account locked,User reports timeout when submitting forms.,email,2025-09-01T06:18:01.104112Z,en,2025-09-01
gamma,c68ebd2f-1f0b-4fb3-b7f1-971c4f915050,App crash on launch,أخطاء عند تسجيل الدخول.,chat,2025-08-28T19:11:01.104112Z,ar,2025-08-28
beta,b9d4d359-faba-423b-bd25-f3ab5c909f56,Refund request,App crashes after update on Android 14.,chat,2025-08-29T00:31:01.104112Z,en,2025-08-29
alpha,7dbbfb8c-8d92-4946-9f3a-e3f33bc52145,Slow performance,فشل الدفع ولا توجد رسالة تأكيد.,phone,2025-08-29T02:26:01.104112Z,ar,2025-08-29


In [0]:
(df.write.format("delta")
   .mode("overwrite")
   .partitionBy("event_date")
   .saveAsTable("de_genai.bronze_tickets_raw"))

spark.table("de_genai.bronze_tickets_raw").limit(5).display()
print("Rows:", spark.table("de_genai.bronze_tickets_raw").count())


tenant_id,ticket_id,subject,body,channel,created_ts,lang,event_date
gamma,6d0ea567-8baa-4d5d-b26c-0d74816e8572,App crash on launch,Payment ke baad confirmation nahi aaya.,email,2025-09-03T03:03:01.104112Z,ur,2025-09-03
beta,c5895ef6-1ed3-4b45-92ac-eb487c75c929,Payment failed,App open nahi ho rahi.,portal,2025-09-03T13:12:01.104112Z,ur,2025-09-03
alpha,429be4dc-eb9d-4e2d-be6d-b6bcb1c26f83,Payment failed,Payment ke baad confirmation nahi aaya.,email,2025-09-03T08:52:01.104112Z,ur,2025-09-03
gamma,7d955b3e-a9c8-4b5b-89ec-bc8609c110bb,App crash on launch,Payment ke baad confirmation nahi aaya.,portal,2025-09-03T13:22:01.104112Z,ur,2025-09-03
gamma,1e42b09f-6be4-461c-84bd-8fbfa5ed20f4,Refund request,أخطاء عند تسجيل الدخول.,chat,2025-09-03T01:28:01.104112Z,ar,2025-09-03


Rows: 1000


In [0]:
from pyspark.sql import functions as F

src = spark.table("de_genai.bronze_tickets_raw")

silver = (src
    .withColumn("created_ts", F.to_timestamp("created_ts"))
    .withColumn("clean_text", F.trim(F.concat_ws(" ", "subject", "body")))
    .dropna(subset=["tenant_id","ticket_id","clean_text"])
)

(silver.write.format("delta")
      .mode("overwrite")
      .saveAsTable("de_genai.silver_tickets_clean"))

spark.table("de_genai.silver_tickets_clean").limit(5).display()
print("Silver rows:", spark.table("de_genai.silver_tickets_clean").count())


tenant_id,ticket_id,subject,body,channel,created_ts,lang,event_date,clean_text
alpha,446f91a5-c26e-43a0-84d9-15c29eaa3dad,Payment failed,أخطاء عند تسجيل الدخول.,email,2025-08-27T22:12:01.104Z,ar,2025-08-27,Payment failed أخطاء عند تسجيل الدخول.
alpha,891d68b5-ef18-4a20-9322-67693199adca,Payment failed,User reports timeout when submitting forms.,email,2025-08-27T20:09:01.104Z,en,2025-08-27,Payment failed User reports timeout when submitting forms.
beta,dd39bd5b-ae18-4c2a-beef-f52d5697ae92,Account locked,App crashes after update on Android 14.,portal,2025-08-27T19:43:01.104Z,en,2025-08-27,Account locked App crashes after update on Android 14.
beta,75a8846a-a225-418e-b013-25d9db84c347,Slow performance,فشل الدفع ولا توجد رسالة تأكيد.,phone,2025-08-27T19:54:01.104Z,ar,2025-08-27,Slow performance فشل الدفع ولا توجد رسالة تأكيد.
beta,24ec88ce-276a-4715-ae18-1250c9503a3d,Login issue,Login par error aata hai.,chat,2025-08-27T20:41:01.104Z,ur,2025-08-27,Login issue Login par error aata hai.


Silver rows: 1000


In [0]:
# Tables
SILVER_CLEAN = "de_genai.silver_tickets_clean"
SILVER_LLM   = "de_genai.silver_tickets_llm"
GOLD_KPIS    = "de_genai.gold_support_kpis"


OPENAI_API_KEY = "paste your key BETWEEN quotes"
OPENAI_MODEL   = "gpt-4o-mini"

# Batch settings
TEST_LIMIT       = 20     # first test run (small)
CHUNK_SIZE       = 150    # run multiple times to finish all rows
SLEEP_BETWEEN_S  = 0.2    # increase if rate-limited

import os, json, time, re, requests
from pyspark.sql import functions as F, types as T

def call_openai(prompt: str) -> str:
    headers = {"Authorization": f"Bearer {OPENAI_API_KEY}", "Content-Type": "application/json"}
    data = {
        "model": OPENAI_MODEL, "temperature": 0.2,
        "messages": [{"role":"user","content": f"""You are a support triage assistant.
Summarize in <= 2 lines.
Classify sentiment: pos | neu | neg.
Classify priority: P1 | P2 | P3 | P4.
Return STRICT JSON: {{"summary":"...","sentiment":"...","priority":"...","topics":["..."]}}
Text:
{prompt[:3500]}"""}]
    }
    try:
        r = requests.post("https://api.openai.com/v1/chat/completions", headers=headers, json=data, timeout=60)
        r.raise_for_status()
        return r.json()["choices"][0]["message"]["content"]
    except Exception:
        return json.dumps({"summary":"", "sentiment":"neu", "priority":"P4", "topics":[]})

def parse_json(s: str):
    try:
        return json.loads(s)
    except Exception:
        m = re.search(r"\{.*\}", s, flags=re.S)
        if m:
            try:
                return json.loads(m.group(0))
            except Exception:
                pass
    return {"summary":"", "sentiment":"neu", "priority":"P4", "topics":[]}


In [0]:
src = spark.table(SILVER_CLEAN).orderBy(F.rand()).limit(TEST_LIMIT)

rows = src.select("tenant_id","ticket_id","created_ts","channel","lang","clean_text").collect()
out = []
for r in rows:
    raw = call_openai(r["clean_text"])
    parsed = parse_json(raw)
    out.append((
        r["tenant_id"], r["ticket_id"], r["created_ts"], r["channel"], r["lang"], r["clean_text"],
        parsed.get("summary",""),
        parsed.get("sentiment","neu"),
        parsed.get("priority","P4"),
        parsed.get("topics",[])
    ))
    time.sleep(SLEEP_BETWEEN_S)

schema = T.StructType([
    T.StructField("tenant_id", T.StringType()),
    T.StructField("ticket_id", T.StringType()),
    T.StructField("created_ts", T.TimestampType()),
    T.StructField("channel", T.StringType()),
    T.StructField("lang", T.StringType()),
    T.StructField("clean_text", T.StringType()),
    T.StructField("summary", T.StringType()),
    T.StructField("sentiment", T.StringType()),
    T.StructField("priority", T.StringType()),
    T.StructField("topics", T.ArrayType(T.StringType()))
])

df_out = spark.createDataFrame(out, schema=schema)

# First run: overwrite to CREATE the table with these 20 rows
(df_out.write.format("delta").mode("overwrite").saveAsTable(SILVER_LLM))

spark.table(SILVER_LLM).limit(10).display()
print("LLM test rows:", spark.table(SILVER_LLM).count())


tenant_id,ticket_id,created_ts,channel,lang,clean_text,summary,sentiment,priority,topics
gamma,95dba66f-eb7a-481a-a048-11d7a44fcee3,2025-08-29T20:21:01.104Z,chat,ur,Account locked Login par error aata hai.,Account is locked and encountering a login error.,neg,P1,"List(account lock, login error)"
gamma,b393d266-8e5c-460f-b8c6-217355a536bf,2025-09-01T18:56:01.104Z,phone,en,Account locked App crashes after update on Android 14.,Account is locked and app crashes after the update on Android 14.,neg,P1,"List(account issue, app crash, Android 14)"
alpha,14c83969-6057-444b-ab59-181e596eed4e,2025-09-02T10:08:01.104Z,portal,en,Feature request Customer asks for SSO integration.,Customer requests integration of Single Sign-On (SSO) feature.,neu,P2,"List(feature request, SSO integration)"
beta,9c37263b-6d05-4bac-b7a4-24aefc811085,2025-09-01T18:14:01.104Z,chat,ur,Login issue Payment ke baad confirmation nahi aaya.,User is experiencing a login issue and has not received confirmation after payment.,neg,P1,"List(login issue, payment confirmation)"
alpha,1a54d214-623f-4260-84f8-a0456ef61e85,2025-08-28T18:09:01.104Z,phone,en,App crash on launch App crashes after update on Android 14.,App crashes on launch after the update to Android 14.,neg,P1,"List(app crash, Android 14, update issues)"
alpha,b38d8117-09fc-4dc1-b7f2-af57abef7c6d,2025-09-02T18:36:01.104Z,email,ur,Slow performance Login par error aata hai.,Experiencing slow performance and login errors.,neg,P2,"List(performance, login error)"
beta,b9d4d359-faba-423b-bd25-f3ab5c909f56,2025-08-29T00:31:01.104Z,chat,en,Refund request App crashes after update on Android 14.,User requests a refund due to app crashes following an update on Android 14.,neg,P1,"List(refund, app crash, Android 14)"
beta,520b653f-491f-4957-b438-61da9331bcf0,2025-08-28T04:13:01.104Z,portal,ur,App crash on launch Login par error aata hai.,App crashes on launch due to a login error.,neg,P1,"List(app crash, login error)"
beta,84636aa7-239c-4fcc-8a1d-cba1b907b221,2025-08-31T19:28:01.104Z,portal,en,Slow performance User reports timeout when submitting forms.,User reports slow performance with timeouts when submitting forms.,neg,P2,"List(performance, timeout, forms)"
gamma,79ef6d98-15b3-42ef-a050-6f4cac9f0952,2025-09-02T21:35:01.104Z,portal,ur,App crash on launch Payment ke baad confirmation nahi aaya.,App crashes on launch and no confirmation received after payment.,neg,P1,"List(app crash, payment issue, confirmation)"


LLM test rows: 20


Running multiple times to enrich 

In [0]:
clean = spark.table(SILVER_CLEAN).select("tenant_id","ticket_id","created_ts","channel","lang","clean_text")
done  = spark.table(SILVER_LLM).select("ticket_id").distinct()

todo = (clean.join(done, on="ticket_id", how="left_anti")
              .orderBy(F.rand())
              .limit(CHUNK_SIZE))

rows = todo.collect()
if not rows:
    print("No remaining rows to enrich — you're done 🎉")
else:
    out = []
    for r in rows:
        raw = call_openai(r["clean_text"])
        parsed = parse_json(raw)
        out.append((
            r["tenant_id"], r["ticket_id"], r["created_ts"], r["channel"], r["lang"], r["clean_text"],
            parsed.get("summary",""),
            parsed.get("sentiment","neu"),
            parsed.get("priority","P4"),
            parsed.get("topics",[])
        ))
        time.sleep(SLEEP_BETWEEN_S)

    df_chunk = spark.createDataFrame(out, schema=spark.table(SILVER_LLM).schema)
    (df_chunk.write.format("delta").mode("append").saveAsTable(SILVER_LLM))

    total = spark.table(SILVER_LLM).count()
    print(f"Appended {len(out)} rows. Total enriched now: {total}. Re-run this cell until it says no remaining rows.")


No remaining rows to enrich — you're done 🎉


In [0]:
tickets = spark.table(SILVER_LLM)

gold = (tickets
    .withColumn("day", F.to_date("created_ts"))
    .groupBy("tenant_id","day")
    .agg(
        F.count("*").alias("ticket_count"),
        F.expr("avg(case when sentiment='neg' then 1 else 0 end)").alias("neg_share"),
        F.expr("sum(case when priority='P1' then 1 else 0 end)").alias("p1_count"),
        F.expr("sum(case when priority='P2' then 1 else 0 end)").alias("p2_count")
    )
)

(gold.write.format("delta").mode("overwrite").saveAsTable(GOLD_KPIS))

spark.table(GOLD_KPIS).orderBy(F.desc("day")).limit(10).display()
print("Gold KPIs ready.")


tenant_id,day,ticket_count,neg_share,p1_count,p2_count
beta,2025-09-03,33,0.9090909090909091,24,9
gamma,2025-09-03,44,0.9090909090909091,28,16
alpha,2025-09-03,30,0.9,20,10
beta,2025-09-02,47,0.9787234042553191,32,15
gamma,2025-09-02,44,0.9318181818181818,33,11
alpha,2025-09-02,44,0.8863636363636364,26,18
alpha,2025-09-01,44,0.9090909090909091,30,14
beta,2025-09-01,50,0.9,34,16
gamma,2025-09-01,57,0.9298245614035088,36,21
alpha,2025-08-31,44,0.8181818181818182,28,15


Gold KPIs ready.


In [0]:
spark.sql("""
CREATE OR REPLACE VIEW de_genai.v_support_overview AS
SELECT day, tenant_id, ticket_count, neg_share, p1_count, p2_count
FROM de_genai.gold_support_kpis
ORDER BY day DESC, tenant_id
""")

spark.sql("""
CREATE OR REPLACE VIEW de_genai.v_recent_tickets AS
SELECT tenant_id, ticket_id, created_ts, sentiment, priority, topics, summary
FROM de_genai.silver_tickets_llm
ORDER BY created_ts DESC
LIMIT 100
""")

print("Views created: de_genai.v_support_overview, de_genai.v_recent_tickets")


Views created: de_genai.v_support_overview, de_genai.v_recent_tickets
